In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from stanza_tokenizer import StanzaTokenizer

In [ ]:
tokenizer = StanzaTokenizer()

In [ ]:
hp_path = '/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/harry_potter/harry_potter_book_4_chapter_1_pages_15-23.txt'
text = open(hp_path).read()

In [ ]:
hp_df = tokenizer.tokenize_to_df(text)
hp_df

In [ ]:
def normalize(text):
    text = text.lower()
    return text
def is_number(token):
    try:
        float(token)
        return True
    except ValueError:
        return False
print(len(hp_df))
hp_df = hp_df[~hp_df['text'].apply(is_number)]
print(len(hp_df))
hp_df['text'] = hp_df['text'].apply(normalize)
hp_df = hp_df[hp_df['pos'] != 'PUNCT']
print(len(hp_df))
hp_df.drop_duplicates(inplace=True)
print(len(hp_df))

In [ ]:
print(len(hp_df))
hp_df = hp_df[hp_df['pos'] != 'PROPN']
print(len(hp_df))

In [ ]:
# hp_df[hp_df.duplicated(subset=['text', 'pos'])].sort_values('text')
hp_df.drop_duplicates(subset=['text', 'lemma', 'pos'], inplace=True)
print(len(hp_df))

In [ ]:
# df = pd.read_csv('token_data.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_20000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_40000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_60000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_80000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_100000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_120000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_140000.tsv', sep='\t')
# df = pd.read_csv('token_data_0_to_160000.tsv', sep='\t')
# df = pd.read_csv('token_data/token_data_0_to_180000.tsv', sep='\t')
df = pd.read_csv('token_data/token_data_0_to_200000.tsv', sep='\t')
len(df), df.columns

In [ ]:
missing_tokens = []
found_tokens = []
for i, row in hp_df.iterrows():
    token_match_df = df[df['token_hash'] == row['token_hash']]
    if len(token_match_df) == 0:
        missing_tokens.append((row['text'], row['lemma'], row['pos'], row['token_hash']))
    else:
        found_tokens.append((row['text'], row['lemma'], row['pos'], row['token_hash']))
f = len(found_tokens)
m = len(missing_tokens)
t = f + m
print(f"Found tokens: {f} ({f/t:.0%}), Missing tokens: {m} ({m/t:.0%})")

In [ ]:
# 20000 tokens total - Found tokens: 765 (67%), Missing tokens: 372 (33%)
# 40000 tokens total - Found tokens: 830 (73%), Missing tokens: 307 (27%)
# 60000 tokens total - Found tokens: 867 (76%), Missing tokens: 270 (24%)
# 80000 tokens total - Found tokens: 883 (78%), Missing tokens: 254 (22%)
# 100000 tokens total - Found tokens: 898 (79%), Missing tokens: 239 (21%)
# 120000 tokens total - Found tokens: 908 (80%), Missing tokens: 229 (20%)
# 140000 tokens total - Found tokens: 924 (81%), Missing tokens: 213 (19%)
# 160000 tokens total - Found tokens: 932 (82%), Missing tokens: 205 (18%)
# 180000 tokens total - Found tokens: 940 (83%), Missing tokens: 197 (17%)
# 200000 tokens total - Found tokens: 949 (83%), Missing tokens: 188 (17%)
X = [20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000, 180000, 200000]
Y = [372, 307, 270, 254, 239, 229, 213, 205, 197, 188]
plt.plot(X, Y, 'bo-', label='Data points')
# let's fit a curve to this (looks like exponential decay)
def model(x, a, b, c):
    return a * np.exp(b * x) + c
popt, pcov = curve_fit(model, X, Y, p0=(400, -0.00001, 150), )
a, b, c = popt
print(f"Fitted parameters: a={a}, b={b}, c={c}")
x_fit = np.linspace(20000, 200000, 100)
y_fit = model(x_fit, *popt)
plt.plot(x_fit, y_fit, 'r--', label='Fitted curve')
plt.xlabel('Number of Tokens in Dataset')
plt.ylabel('Number of Missing Tokens')
plt.title('Missing Tokens vs. Dataset Size')
plt.legend()

In [ ]:
missing_tokens

In [ ]:
missing_groups = []
found_groups = []
temp = hp_df.drop_duplicates(subset=['group_hash'])
for i, row in temp.iterrows():
    group_match_df = df[df['group_hash'] == row['group_hash']]
    if len(group_match_df) == 0:
        missing_groups.append((row['text'], row['lemma'], row['pos'], row['group_hash']))
    else:
        found_groups.append((row['text'], row['lemma'], row['pos'], row['group_hash']))
fg = len(found_groups)
mg = len(missing_groups)
tg = fg + mg
print(f"Found groups: {fg} ({fg/tg:.0%}), Missing groups: {mg} ({mg/tg:.0%})")

In [ ]:
# 20000 tokens total - Found groups: 674 (76%), Missing groups: 208 (24%)
# 40000 tokens total - Found groups: 721 (82%), Missing groups: 161 (18%)
# 60000 tokens total - Found groups: 746 (85%), Missing groups: 136 (15%)
# 80000 tokens total - Found groups: 758 (86%), Missing groups: 124 (14%)
# 100000 tokens total - Found groups: 763 (87%), Missing groups: 119 (13%)
# 120000 tokens total - Found groups: 768 (87%), Missing groups: 114 (13%)
# 140000 tokens total - Found groups: 776 (88%), Missing groups: 106 (12%)
# 160000 tokens total - Found groups: 779 (88%), Missing groups: 103 (12%)
# 180000 tokens total - Found groups: 783 (89%), Missing groups: 99 (11%)
# 200000 tokens total - Found groups: 790 (90%), Missing groups: 92 (10%)
X = [20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000, 180000, 200000]
Y = [208, 161, 136, 124, 119, 114, 106, 103, 99, 92]
plt.figure()
plt.plot(X, Y, 'bo-', label='Data points')
# let's fit a curve to this (looks like exponential decay)
def model(x, a, b, c):
    return a * np.exp(b * x) + c
popt, pcov = curve_fit(model, X, Y, p0=(300, -0.00001, 80))
a, b, c = popt
print(f"Fitted parameters: a={a}, b={b}, c={c}")
x_fit = np.linspace(20000, 200000, 100)
y_fit = model(x_fit, *popt)
plt.plot(x_fit, y_fit, 'r--', label='Fitted curve')
plt.xlabel('Number of Tokens in Dataset')
plt.ylabel('Number of Missing Groups')
plt.title('Missing Groups vs. Dataset Size')
plt.legend()

In [ ]:
missing_groups